<a href="https://colab.research.google.com/github/manyajain2435/AAI_primer/blob/main/LLM_Session_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Session 3: The ReAct Loop (Reason + Act) - The Heart of Agency

Goal: Move from single tool call to multi-step reasoning.

In [4]:
!pip install groq

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
from google.colab import userdata

api_key = userdata.get("gsk_WORKSHOP_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")

        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

In [5]:
# Tool Definitions
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response


In [8]:
import json
from google.colab import userdata
from groq import Groq

# Get Groq API key from Colab Secrets
api_key = userdata.get("gsk_WORKSHOP_KEY")

# Create Groq client
client = Groq(api_key=api_key)


def run_agent(user_query):
    messages = [
        {"role": "user", "content": user_query}
    ]

    # Agentic loop
    while True:

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tool_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message

        # Add LLM response to conversation
        messages.append(response_message)

        # Check whether the LLM wants to use a tool
        if response_message.tool_calls:

            for tool_call in response_message.tool_calls:

                function_name = tool_call.function.name

                function_args = json.loads(
                    tool_call.function.arguments
                )

                # Execute the requested Python function
                if function_name == "get_weather":

                    result = get_weather(
                        function_args.get("city")
                    )

                    # Send tool result back to LLM
                    messages.append({
                        "role": "tool",
                        "content": result,
                        "tool_call_id": tool_call.id
                    })

        else:
            return response_message.content

In [9]:
print(run_agent("Is it warmer in Mumbai or Delhi right now?"))

Based on the responses, it is warmer in Mumbai (32C) compared to Delhi (28C) right now. However, it's worth noting that the humidity in Mumbai is high, while Delhi is experiencing smoggy conditions.
